# Phase 3 RF-QRC time multiplexing

This is the physics-clean alternative to a two-QRC hybrid.

Question: can the same 6-qubit RF-QRC ring reservoir recover useful calibration/tail balance by reading out several virtual time nodes from one encoded state?

Mechanism:

```text
encode input -> ring entangle -> second encode
then repeat fixed random layer and collect <Z_i>, <Z_i Z_j> after each virtual node
```

This keeps one reservoir and one input encoding. It only changes the observable/readout horizon.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == 'qpitome-qrc-volatility' and (p / 'scripts').exists() and (p / 'src').exists():
            return p
    raise RuntimeError('Open this notebook from inside qpitome-qrc-volatility.')

ROOT = find_repo_root()
os.chdir(ROOT)
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('Repo root:', ROOT)


## Run bounded virtual-node sweep

In [ ]:
script = ROOT / 'scripts' / 'run_phase3_rf_qrc_time_multiplexing.py'
if not script.exists():
    raise FileNotFoundError(script)

cmd = [
    sys.executable, str(script),
    '--leak', '0.3',
    '--input-scale', str(np.pi / 3),
    '--random-scale', '0.35',
    '--layer-scale', '1.0',
    '--virtual-nodes', '1', '2', '3', '5',
    '--ridge-alphas', '1000', '3000', '10000',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Load outputs

In [ ]:
metrics_path = TABLES / 'phase3_rf_qrc_time_multiplex_metrics.csv'
summary_path = TABLES / 'phase3_rf_qrc_time_multiplex_test_summary.csv'
pred_path = TABLES / 'phase3_rf_qrc_time_multiplex_predictions.csv'

metrics = pd.read_csv(metrics_path)
summary = pd.read_csv(summary_path)
preds = pd.read_csv(pred_path)

display(summary.sort_values(['q95_f1', 'qlike'], ascending=[False, True]))
display(preds.head())


## Selection table

In [ ]:
selection = summary.copy()
selection['passes_tm_gate'] = (
    (selection['qlike'] < -1.5) &
    (selection['q95_f1'] >= 0.35) &
    (selection['pred_std'] >= 0.05) &
    (selection['corr'] >= 0.50)
)

selection_path = TABLES / 'phase3_rf_qrc_time_multiplex_selection_table.csv'
selection.to_csv(selection_path, index=False)
print('Saved:', selection_path)
display(selection.sort_values(['passes_tm_gate', 'q95_f1', 'qlike'], ascending=[False, False, True]))


## Compare against existing RF-QRC ring / Phase 2 / ESN table if available

In [ ]:
context_path = TABLES / 'phase3_clean_raw_forecast_comparison.csv'
if context_path.exists():
    context = pd.read_csv(context_path)
    display(context)
else:
    print('No context table found:', context_path)

best = selection.sort_values(['passes_tm_gate', 'q95_f1', 'qlike'], ascending=[False, False, True]).iloc[0]
print('Best time-multiplex candidate:')
print(best.to_string())


## Figures

In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

# Metric trends by virtual nodes.
for metric in ['qlike', 'corr', 'pred_std', 'q95_f1']:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for alpha, g in summary.groupby('ridge_alpha'):
        g = g.sort_values('virtual_nodes')
        ax.plot(g['virtual_nodes'], g[metric], marker='o', label=f'alpha={int(alpha)}')
    ax.set_xlabel('virtual nodes')
    ax.set_ylabel(metric)
    ax.set_title(f'Time multiplexing: {metric}')
    ax.legend()
    savefig(FIGURES / f'phase3_rf_qrc_time_multiplex_{metric}_trend.png')

# QLIKE vs q95 F1 tradeoff.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(selection['qlike'], selection['q95_f1'])
for _, row in selection.iterrows():
    label = f"vn={int(row['virtual_nodes'])}, a={int(row['ridge_alpha'])}"
    ax.annotate(label, (row['qlike'], row['q95_f1']), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.axhline(0.35, linestyle='--', linewidth=1)
ax.axvline(-1.5, linestyle='--', linewidth=1)
ax.set_xlabel('repo-style QLIKE; lower is better')
ax.set_ylabel('q95 amplitude F1')
ax.set_title('RF-QRC time multiplexing: QLIKE vs q95 F1')
savefig(FIGURES / 'phase3_rf_qrc_time_multiplex_qlike_vs_q95_f1.png')

# Test forecast traces for best candidate.
best_col = best['run_name'] + '_pred'
test = preds[preds['split'].astype(str).str.lower().eq('test')].copy()
x = pd.to_datetime(test['date'], errors='coerce')
if x.isna().all():
    x = np.arange(len(test))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, test['actual_future_rv_20d'], label='actual', linewidth=2)
ax.plot(x, test[best_col], label=best_col.replace('_pred', ''))
ax.set_title('Best RF-QRC time-multiplex forecast on test period')
ax.set_ylabel('future RV 20d')
ax.legend()
savefig(FIGURES / 'phase3_rf_qrc_time_multiplex_best_forecast.png')


## Interpretation

In [ ]:
print('Gate: qlike < -1.5, q95_f1 >= 0.35, pred_std >= 0.05, corr >= 0.50')
display(selection[['run_name', 'virtual_nodes', 'ridge_alpha', 'qlike', 'corr', 'pred_std', 'q95_f1', 'top20_pred_actual_ratio', 'effective_rank', 'passes_tm_gate']].sort_values(['passes_tm_gate', 'q95_f1', 'qlike'], ascending=[False, False, True]))

if bool(selection['passes_tm_gate'].any()):
    print('Time multiplexing gives a one-reservoir route worth following.')
else:
    print('Time multiplexing does not pass the current gate; treat as another diagnostic result.')
